# Traditional ML Pipeline (Task 3 & 4)

This notebook trains and evaluates **traditional machine learning** vegetation classifiers (no deep learning) by benchmarking:

- Models: `SVM`, `RandomForest`, `KNN`
- Feature extractors: `Color Histogram`, `Color Moment`, `HOG`, `GLCM`, `LBP`, `SIFT` (and optional `SURF` if available)
- Feature combinations (single and fused), then selects the best by validation accuracy

It evaluates the best pipeline on a held-out test set and writes required artifacts (confusion matrix, per-class metrics, qualitative examples, and report).

## Data Preparation
```bash
# Data extraction
python ./extract_grouped.py --interval-ms 1000 --clean;

# Data augmentation
python ./expand_dataset_offline.py --source-subdir extracted_grouped --output-subdir extracted_group_augmented --target-fold 25.0 --clean --equalize-classes

# Data summary
python info.py --root ./polyu_vegetation/raw_grouped --target ./polyu_vegetation/extracted_group_augmented
```

```bash
## Example Summary for dataset used in training:
class                     root  target  delta  fold
---------------------------------------------------
anthurium andraeanum         8     200    192  25.00x
arachnothryx leucophylla    11     200    189  18.18x
camellia japonica           13     200    187  15.38x
codiaeum variegatum         14     200    186  14.29x
ficus microcarpa            25     200    175  8.00x
juniperus chinensis         13     200    187  15.38x
livistona chinensis         14     200    186  14.29x
rhododendron                29     200    171  6.90x
---------------------------------------------------
TOTAL                      127    1600   1473  12.60x
```

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np

from dataset_loader import (
    PipelineConfig,
    PipelineDataset,
    load_pipeline_dataset,
    extract_base_feature_sets,
    collect_samples,
    load_images,
    reduce_base_features_with_pca,
)
from joblib import dump
from train import run_traditional_ml_pipeline
from summary import write_traditional_ml_outputs
project_root = Path.cwd()


In [ ]:
# Data loading

cfg = PipelineConfig(
    data_root=project_root / "polyu_vegetation",
    dataset_subdir="extracted_group_augmented",  # use offline-augmented dataset
    output_dir=project_root / "ml_results",
    seed=42,
    img_size=128,
    show_progress=True,
)

# Load once, then reuse in later cells.
dataset = load_pipeline_dataset(cfg)
print(f"Loaded dataset with {len(dataset.images)} image samples from: {dataset.dataset_dir}")

In [ ]:
# Feature extraction

# fit_new_reduction=True: pipeline fits PCA per CV fold (honest val) + one PCA on full train pool for final models/test.
# fit_new_reduction=False: load ml_results/models/pcas.joblib and reduce here (legacy; CV does not refit PCA).
fit_new_reduction = True
n_pca_components = 400
features_to_reduce = ["hog"]

print(
    f"PCA n={n_pca_components} on {', '.join(features_to_reduce)} "
    "(per-fold fit inside training when fit_new_reduction=True)"
)

# Proposed combinations
proposed_feature_combinations = (
    "color_hist+color_moment",
    "color_hist+sift",
    "color_hist+lbp",
    "color_hist+glcm",
    "color_hist+hog",
    "color_hist+color_moment+lbp",
    "color_hist+color_moment+hog",
    "color_hist+sift+lbp",
    "color_hist+sift+hog",
    "color_hist+sift+lbp+hog",
    "color_hist+color_moment+sift+lbp+hog",
    "color_hist+sift+glcm",
    "color_hist+sift+hog",
    "color_hist+lbp+glcm",
    "color_hist+glcm+hog",
)

# Test dataset (kept separate so you can reuse/pass it around)
test_dir = project_root / "polyu_vegetation" / "test"
test_paths, test_labels = collect_samples(
    dataset_dir=test_dir,

    max_examples_per_group=max(1, cfg.max_examples_per_group),
)
test_images = np.asarray(

    load_images(np.asarray(test_paths, dtype=object), size=cfg.img_size),

    dtype=np.uint8,
)

train_images = list(dataset.images)

unreduced_base_features = extract_base_feature_sets(
    train_images,

    img_size=cfg.img_size,
)

if fit_new_reduction:
    train_base_for_pipeline = unreduced_base_features
    reduced_base_features, fitted_pcas = reduce_base_features_with_pca(
        unreduced_base_features,
        n_components=n_pca_components,
        features_to_reduce=features_to_reduce,
    )
else:
    from joblib import load

    pcas_file = project_root / "ml_results" / "models" / "pcas.joblib"
    fitted_pcas = load(pcas_file)
    train_base_for_pipeline, _ = reduce_base_features_with_pca(
        unreduced_base_features,
        n_components=n_pca_components,
        features_to_reduce=features_to_reduce,
        fitted_pcas=fitted_pcas,
    )

feature_names = list(unreduced_base_features.keys())

print(f"Available features: {', '.join(feature_names)}")


test_dataset = PipelineDataset(
    images=test_images,

    labels=np.asarray(test_labels, dtype=object),
    dataset_dir=test_dir,

    paths=np.asarray(test_paths, dtype=object),
)

print(f"Loaded test dataset with {len(test_dataset.images)} samples from: {test_dir}")

test_images = list(test_dataset.images)

test_base_features_unreduced = extract_base_feature_sets(
    test_images,

    img_size=cfg.img_size,
)

if fit_new_reduction:
    test_base_for_pipeline = test_base_features_unreduced
else:
    test_base_for_pipeline, _ = reduce_base_features_with_pca(
        test_base_features_unreduced,
        n_components=n_pca_components,
        features_to_reduce=features_to_reduce,
        fitted_pcas=fitted_pcas,
    )

In [ ]:
# Train models using the preloaded dataset.

save_model = True # save each model with its highest test accuracy version everytime
clean_model_directory = True

run_data = run_traditional_ml_pipeline(
    cfg,
    train_dataset=dataset,
    train_base_features=train_base_for_pipeline,
    test_dataset=test_dataset,
    test_base_features=test_base_for_pipeline,
    save_model=save_model,
    clean_model_directory=clean_model_directory,
    features_combinations=proposed_feature_combinations,
    pca_n_components=n_pca_components if fit_new_reduction else None,
    pca_features_to_reduce=features_to_reduce if fit_new_reduction else None,
    fitted_pcas_full_pool=fitted_pcas,
    # features_combinations=["color_hist+hog", "color_hist+sift+hog"]
)
if fitted_pcas is not None:
    _ = dump(fitted_pcas, project_root / "ml_results" / "models" / "pcas.joblib")

# Save Task 3/4 outputs (confusion matrix, report, JSON summaries, examples).
summary_data = write_traditional_ml_outputs(run_data, test_size=len(test_dataset.labels))

# Best model per algorithm across all feature combinations.
svm_models = run_data.train_results_by_model["svm"]

rf_models = run_data.train_results_by_model["random_forest"]

knn_models = run_data.train_results_by_model["knn"]

print("----------------------------------------------------------------")
print("Top 3 models by validation accuracy:")
print("----------------------------------------------------------------")

for i, model in enumerate(run_data.train_results[:3]):
    print(f"{1+i}. {model.model_name}: {model.feature_name} \nTest accuracy: {model.test_accuracy}, Val accuracy: {model.val_accuracy}, Train accuracy: {model.train_accuracy}, Time: {model.train_seconds}s")

print("----------------------------------------------------------------")
print("Top 3 models by test accuracy:")
for i, model in enumerate(run_data.train_results_sorted_by_test_accuracy[:3]):
    print(f"{1+i}. {model.model_name}: {model.feature_name} \nTest accuracy: {model.test_accuracy}, Val accuracy: {model.val_accuracy}, Train accuracy: {model.train_accuracy}, Time: {model.train_seconds}s")

print("----------------------------------------------------------------")
print("Top 3 for each model:")

for models in [svm_models, rf_models, knn_models]:
    model_name = models[0].model_name
    print(f"{model_name}: ")
    for i, m in enumerate(models[:3]):
        print(f"\t{1+i}. {m.feature_name} \n\tTest accuracy: {m.test_accuracy}, Val accuracy: {m.val_accuracy}, Train accuracy: {m.train_accuracy}, Time: {m.train_seconds}s")
    print("")

print("----------------------------------------------------------------")


print(f"\nTotal evaluated pairs: {len(run_data.train_results)}")

## Feature Extraction Pipeline Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

from dataset_loader import reduce_base_features_with_pca

if "run_data" in dir() and getattr(run_data, "fitted_pcas", None) is not None:
    reduced_features, _ = reduce_base_features_with_pca(
        unreduced_base_features,
        fitted_pcas=run_data.fitted_pcas,
        n_components=n_pca_components,
        features_to_reduce=features_to_reduce,
    )
else:
    reduced_features, _ = reduce_base_features_with_pca(
        unreduced_base_features,
        n_components=n_pca_components,
        features_to_reduce=features_to_reduce,
    )

# Dimensions before/after PCA
names = list(unreduced_base_features.keys())
dims_before = [unreduced_base_features[n].shape[1] for n in names]
dims_after = [reduced_features[n].shape[1] for n in names]

fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [1.2, 1]})

# --- Plot 1: Bar chart of feature block dimensions
ax1 = axes[0]
x = np.arange(len(names))
width = 0.35
bars1 = ax1.bar(x - width / 2, dims_before, width, label="Before PCA", color="steelblue", alpha=0.8)
bars2 = ax1.bar(x + width / 2, dims_after, width, label=f"After PCA (max {400})", color="coral", alpha=0.8)
ax1.set_ylabel("Dimensions")
ax1.set_title("Feature Block Dimensions: Before vs After PCA per Block")
ax1.set_xticks(x)
ax1.set_xticklabels(names)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)
for b in bars1:
    ax1.annotate(str(b.get_height()), xy=(b.get_x() + b.get_width() / 2, b.get_height()),
                 ha="center", va="bottom", fontsize=8, rotation=90)
for b in bars2:
    ax1.annotate(str(b.get_height()), xy=(b.get_x() + b.get_width() / 2, b.get_height()),
                 ha="center", va="bottom", fontsize=8, rotation=90)

# --- Plot 2: Concatenation diagram for example combo (color_hist+lbp+sift)
example_combo = "color_hist+sift+lbp+hog"
parts = [p.strip() for p in example_combo.split("+") if p.strip()]
if all(p in reduced_features for p in parts):
    dims = [reduced_features[p].shape[1] for p in parts]
    total = sum(dims)
    ax2 = axes[1]
    left = 0
    colors = plt.cm.Set3(np.linspace(0, 1, len(parts)))
    for i, (p, d) in enumerate(zip(parts, dims)):
        w = d / total
        rect = FancyBboxPatch((left, 0.2), w, 0.6, boxstyle="round,pad=0.01", 
                              facecolor=colors[i], edgecolor="black", linewidth=1)
        ax2.add_patch(rect)
        ax2.text(left + w / 2, 0.5, f"{p}\n{d}d", ha="center", va="center", fontsize=10)
        left += w
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.set_aspect("equal")
    ax2.axis("off")
    ax2.set_title(f"Concatenated Feature Vector: {example_combo} = {total} total dims")
else:
    axes[1].text(0.5, 0.5, f"Example combo '{example_combo}' not available\n(available: {list(reduced_features.keys())})",
                 ha="center", va="center", fontsize=11)
    axes[1].axis("off")

plt.tight_layout()
plt.show()

# Confusion Matrix

In [ ]:
# Top-3 confusion matrix heatmaps loaded from saved outputs (test set only)
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

results_dir = Path.cwd() / "ml_results"
summary_path = results_dir / "run_summary.json"
show_normalized = True

if not summary_path.exists():
    raise FileNotFoundError(f"Missing file: {summary_path}. Run the training cell first.")

run_summary = json.loads(summary_path.read_text(encoding="utf-8"))
top_pipelines = run_summary.get("top_pipelines", [])

if not top_pipelines:
    raise ValueError("No top pipeline confusion-matrix metadata found in run_summary.json.")

n_rows = len(top_pipelines)
fig, axes = plt.subplots(n_rows, 1, figsize=(10, 8 * n_rows), squeeze=False)

for row_idx, info in enumerate(top_pipelines):
    rank = info["rank"]
    pipeline_name = info["pipeline"]
    if show_normalized:
        test_file_key = "test_confusion_matrix_normalized_csv"
    else:
        test_file_key = "test_confusion_matrix_csv"
    test_file = info[test_file_key]
    test_path = results_dir / test_file

    if not test_path.exists():
        raise FileNotFoundError(
            f"Missing file: {test_path}. Re-run the training/output cell first."
        )

    test_cm_df = pd.read_csv(test_path, index_col=0)
    classes = list(test_cm_df.index)
    cm = test_cm_df.to_numpy(dtype=float)

    ax = axes[row_idx, 0]
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(
        xticks=np.arange(len(classes)),
        yticks=np.arange(len(classes)),
        xticklabels=classes,
        yticklabels=classes,
        ylabel="True label",
        xlabel="Predicted label",
        title=f"Top {rank}: {pipeline_name}\nTest Confusion Matrix (Row-Normalized)",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    threshold = cm.max() * 0.7 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm[i, j]
            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                color="white" if val > threshold else "black",
            )

fig.tight_layout()
plt.show()

In [ ]:
# Show qualitative examples: 1 correct and 1 wrong per class from qualitative_examples_per_class.csv
import csv
import cv2
import random
from pathlib import Path
import matplotlib.pyplot as plt

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}
extracted_grouped_dir = project_root / "polyu_vegetation" / "extracted_grouped"


def _random_sample_from_class(class_name: str) -> Path | None:
    """Pick a random image from extracted_grouped/<class_name>/."""
    class_dir = extracted_grouped_dir / class_name
    if not class_dir.exists() or not class_dir.is_dir():
        return None
    images = [p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
    return random.choice(images) if images else None


output_dir = project_root / "ml_results"
per_class_csv = output_dir / "qualitative_examples_per_class.csv"


def load_per_class_csv(path: Path) -> list[dict]:
    """Load per-class examples: class, correct_path, correct_conf, wrong_path, wrong_pred, wrong_conf."""
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows


per_class = load_per_class_csv(per_class_csv)
print(f"Loaded {len(per_class)} classes from {per_class_csv}")

# If no loaded class from per_class_csv, load the latest model from ml_results/models and run inferences
if not per_class:
    from joblib import load
    from dataset_loader import (
        build_feature_sets,
        extract_base_feature_sets,
        reduce_base_features_with_pca,
        collect_samples,
        load_images,
    )
    from summary import pick_examples, margin_confidence

    models_dir = project_root / "ml_results" / "models"
    test_dir = project_root / "polyu_vegetation" / "test"
    img_size = 128

    def _infer_feature_combo_from_model_name(model_path: str) -> str:
        stem = Path(model_path).stem
        if "_" in stem and stem.split("_", 1)[0].isdigit():
            stem = stem.split("_", 1)[1]
        for model_name in ("random_forest", "svm", "knn"):
            prefix = f"{model_name}_"
            if stem.startswith(prefix):
                return stem[len(prefix) :]
        return ""

    def _find_latest_model() -> Path | None:
        candidates = sorted(
            models_dir.glob("*"),
            key=lambda p: p.stat().st_mtime if p.is_file() else 0,
            reverse=True,
        )
        for p in candidates:
            if p.suffix.lower() in (".pkl", ".joblib") and p.name != "pcas.joblib":
                return p
        return None

    latest_model = _find_latest_model()
    pcas_file = models_dir / "pcas.joblib"
    
    if latest_model and test_dir.exists() and pcas_file.exists():
        try:
            model = load(latest_model)
            feature_combo = _infer_feature_combo_from_model_name(str(latest_model))
            if not feature_combo:
                print("[WARN] Could not infer feature combo from model name; skipping inference fallback.")
            else:
                fitted_pcas = load(pcas_file)

                test_paths, test_labels = collect_samples(
                    dataset_dir=test_dir,
                    max_examples_per_group=max(1, 99999),
                )
                test_images = list(load_images(np.asarray(test_paths, dtype=object), size=img_size))

                base_features = extract_base_feature_sets(test_images, img_size=img_size)
                print(base_features.keys())
                base_features, _ = reduce_base_features_with_pca(
                    base_features,
                    n_components=400,
                    features_to_reduce=["hog"],
                    fitted_pcas=fitted_pcas,
                )
                print(len(base_features["hog"][0]))
                feature_sets = build_feature_sets(
                    test_images,
                    img_size=img_size,
                    feature_combinations=(feature_combo,),
                    base_features=base_features,
                )
                x = feature_sets[feature_combo]
                y_pred = model.predict(x)
                y_true = np.asarray(test_labels, dtype=object)
                confidences = margin_confidence(model, x)

                _, _, per_class_tuples = pick_examples(
                    test_paths, y_true, y_pred, confidences, one_per_class=True
                )
                per_class = []
                for cls, correct_row, wrong_row in per_class_tuples:
                    row = {"class": cls, "correct_image_path": "", "correct_confidence": "", "wrong_image_path": "", "wrong_pred_label": "", "wrong_confidence": ""}
                    if correct_row:
                        path, _, _, conf = correct_row
                        row["correct_image_path"] = str(path)
                        row["correct_confidence"] = f"{conf:.6f}"
                    if wrong_row:
                        path, _, pred, conf = wrong_row
                        row["wrong_image_path"] = str(path)
                        row["wrong_pred_label"] = str(pred)
                        row["wrong_confidence"] = f"{conf:.6f}"
                    per_class.append(row)
                print(f"Ran inference with {latest_model.name}; populated {len(per_class)} classes.")
        except Exception as e:
            print(f"[WARN] Inference fallback failed: {e}")
    elif not latest_model:
        print("[WARN] No model found in ml_results/models; cannot run inference fallback.")
    elif not pcas_file.exists():
        print("[WARN] pcas.joblib not found in ml_results/models; models expect PCA-reduced features. Run training first.")
    else:
        print("[WARN] Test dir not found; cannot run inference fallback.")

n_classes = len(per_class)
if n_classes == 0:
    print("No per-class data available. Run training first or ensure ml_results/models contains a trained model.")
else:
    n_cols = 3  # Correct | Wrong | Random sample from predicted class
    fig, axes = plt.subplots(n_classes, n_cols, figsize=(4 * n_cols, 3 * n_classes))
    if n_classes == 1:
        axes = axes.reshape(1, -1)

    for row_idx, row in enumerate(per_class):
        cls = row["class"]
        # Column 0: Correct example
        ax = axes[row_idx, 0]
        img_path = row.get("correct_image_path", "").strip()
        if not img_path:
            ax.text(0.5, 0.5, "No correct example", ha="center", va="center")
            ax.set_title(f"Correct | {cls}")
        else:
            img = cv2.imread(str(img_path))
            if img is None:
                ax.text(0.5, 0.5, f"Failed to load\n{Path(img_path).name}", ha="center", va="center")
            else:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            conf = row.get("correct_confidence", "")
            conf_val = float(conf) if conf else 0.0
            ax.set_title(f"Correct | {cls}\nconf: {conf_val:.3f}", fontsize=9)
        ax.axis("off")
        axes[row_idx, 0].set_ylabel(cls, fontsize=10, rotation=0, labelpad=30)

        # Column 1: Wrong (misclassified) example
        ax = axes[row_idx, 1]
        img_path = row.get("wrong_image_path", "").strip()
        wrong_pred = row.get("wrong_pred_label", "").strip()
        if not img_path:
            ax.text(0.5, 0.5, "No wrong example", ha="center", va="center")
            ax.set_title(f"Wrong | {cls}")
        else:
            img = cv2.imread(str(img_path))
            if img is None:
                ax.text(0.5, 0.5, f"Failed to load\n{Path(img_path).name}", ha="center", va="center")
            else:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            conf = row.get("wrong_confidence", "")
            conf_val = float(conf) if conf else 0.0
            ax.set_title(f"Wrong | {cls}\npred: {wrong_pred} | conf: {conf_val:.3f}", fontsize=9)
        ax.axis("off")

        # Column 2: Random sample from predicted class (extracted_grouped)
        ax = axes[row_idx, 2]
        if wrong_pred:
            sample_path = _random_sample_from_class(wrong_pred)
            if sample_path is None:
                ax.text(0.5, 0.5, f"No {wrong_pred}\nin extracted_grouped", ha="center", va="center")
                ax.set_title(f"Sample of predicted class | {wrong_pred}")
            else:
                img = cv2.imread(str(sample_path))
                if img is None:
                    ax.text(0.5, 0.5, f"Failed to load\n{sample_path.name}", ha="center", va="center")
                else:
                    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                ax.set_title(f"Sample of predicted class | {wrong_pred}")
        else:
            ax.text(0.5, 0.5, "N/A (no wrong pred)", ha="center", va="center")
            ax.set_title("Sample of predicted class")
        ax.axis("off")

    fig.suptitle("Qualitative examples: 1 correct + 1 wrong per class + sample from predicted class (extracted_grouped)", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Show saved report of models' performance from ml_results
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display

results_dir = Path.cwd() / "ml_results"
run_summary_path = results_dir / "run_summary.json"
candidate_scores_path = results_dir / "train_results.json"
report_path = results_dir / "task3_task4_report.md"

for required in (run_summary_path, candidate_scores_path, report_path):
    if not required.exists():
        raise FileNotFoundError(f"Missing file: {required}. Run the training cell first.")

with run_summary_path.open("r", encoding="utf-8") as f:
    run_summary = json.load(f)
with candidate_scores_path.open("r", encoding="utf-8") as f:
    candidate_scores = json.load(f)

scores_df = pd.DataFrame(candidate_scores).sort_values("test_accuracy", ascending=False)

print("Saved run summary")
print(f"- Dataset: {run_summary.get('dataset_dir', 'N/A')}")
print(f"- Best pipeline: {run_summary.get('best_pipeline', 'N/A')}")
print(f"- Test accuracy: {run_summary.get('test_accuracy', 'N/A')}")

display(scores_df[["pipeline", "val_accuracy", "test_accuracy", "train_accuracy", "train_seconds"]])

report_text = report_path.read_text(encoding="utf-8")
display(Markdown(report_text))

In [ ]:
# Image example from each class from ./polyu_vegetation/extracted
import matplotlib.pyplot as plt
import cv2

# One image per class from the loaded dataset
classes = sorted(np.unique(dataset.labels).tolist())
example_indices = {cls: np.where(dataset.labels == cls)[0][0] for cls in classes}

n_cols = 4
n_rows = (len(classes) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
axes = axes.flatten()
for i, cls in enumerate(classes):
    idx = example_indices[cls]
    img = dataset.images[idx]
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(rgb)
    axes[i].set_title(cls, fontsize=9)
    axes[i].axis("off")
for j in range(len(classes), len(axes)):
    axes[j].axis("off")
fig.suptitle("One image example per class")
plt.tight_layout()
plt.show()